# Instella Reasoning — GPU Experiment Suite (resumable + Drive-persisted)

**Cell 1** sets up and runs the suite. It survives interruptions two ways:
- **Resumable**: each (model, benchmark) already scored is skipped.
- **Drive-persisted**: results are written to `MyDrive/instella-gpu-suite-L<LIMIT>/`, so a
  full Colab VM recycle (which wipes `/content`) does NOT lose finished work.

So if Colab disconnects, just **re-run Cell 1** — it remounts Drive, fast-forwards past
everything already done, and continues. Expect ~30 min per (model, benchmark) at LIMIT=200,
so the full 4 models × 6 benchmarks is ~12 h of GPU time spread over as many sessions as it takes.

**Cell 2** reads results straight from Drive — no download needed.

Prereqs: Runtime → GPU · Secrets (🔑): `github` (GitHub PAT, repo read) and `HF_TOKEN`,
both Notebook-access ON. The first run asks permission to mount Google Drive — accept it.

In [ ]:
LIMIT = 200   # 20 = fast sanity pass · 200 = the real smoke run. Re-run this cell to resume.

# ---- persist results to Drive so they survive a full VM recycle ----
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = f"/content/drive/MyDrive/instella-gpu-suite-L{LIMIT}"   # results live here, on Drive

# ---- setup (idempotent: clone or pull, install only if missing) ----
import os, subprocess, glob, json
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("github")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

if not os.path.isdir("/content/Instella-Reasoning"):
    !git clone -b agent/cpu-experiment-suite https://$GITHUB_TOKEN@github.com/GIND123/Instella-Reasoning /content/Instella-Reasoning
%cd /content/Instella-Reasoning
!git pull

if subprocess.run(["which", "instella-reasoning"], capture_output=True).returncode != 0:
    !pip install -q -e ".[hf,retrieval,viz,stats]"
    !pip install -q "bitsandbytes>=0.46.1"

# ---- run the suite (resumable; writes to Drive RUN_DIR) ----
!bash experiments/run_gpu_suite.sh {LIMIT} "{RUN_DIR}"

# ---- accuracy quick-look ----
print(f"\n=== accuracy per (model x benchmark), LIMIT={LIMIT} ===")
for f in sorted(glob.glob(f"{RUN_DIR}/scores/*__*.jsonl")):
    if f.endswith("__ALL.jsonl"):
        continue
    rows = [json.loads(l) for l in open(f) if l.strip()]
    if rows:
        acc = sum(r["correct"] for r in rows) / len(rows)
        print(f"{f.split('/')[-1]:50} {len(rows):4} items  acc={acc:.3f}")

In [ ]:
# Results already persist on Drive under MyDrive/instella-gpu-suite-L<LIMIT>/.
# To also grab a local zip:
LIMIT = 200
!cd /content/drive/MyDrive && zip -qr /content/gpu_results.zip instella-gpu-suite-L{LIMIT} && echo zipped
from google.colab import files
files.download("/content/gpu_results.zip")